## Prepare Env

In [ ]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
from delta import configure_spark_with_delta_pip

In [ ]:
# Define S3 storage
obj_storage_access_key = os.getenv('OBJ_STORAGE_ACCESS_KEY', 'demo-access-key')
obj_storage_secret_key = os.getenv('OBJ_STORAGE_SECRET_KEY', 'demo-secret-key')
obj_storage_endpoint = os.getenv('OBJ_STORAGE_ENDPOINT', 'http://localhost:9000')

In [ ]:
yaml_content = f"""source:
  type: "delta-lake"
  config:
    base_path: "s3://warehouse/bronze/opensanctions_entities.delta"
    s3:
      aws_config:
        aws_access_key_id: "{obj_storage_access_key}"
        aws_secret_access_key: "{obj_storage_secret_key}"
        aws_endpoint_url: "{obj_storage_endpoint}"
        aws_region: us-west-2

sink:
  type: "datahub-rest"
  config:
    server: "http://localhost:8080"
"""

with open('delta.s3.dhub.yaml', 'w') as file:
    file.write(yaml_content)


In [ ]:
! datahub ingest -c delta.s3.dhub.yaml

## Ingestion
### 1. Layer files to layer bronze
Write files which are in layer files to delta table in layer bronze

In [ ]:
# Create a Spark session
builder = SparkSession.builder \
    .appName("Example_OpenSanctions_Datahub") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.extraListeners","datahub.spark.DatahubSparkListener") \
    .config("spark.datahub.rest.server", "http://localhost:8080") 

spark = configure_spark_with_delta_pip(builder, extra_packages=['org.apache.hadoop:hadoop-aws:3.3.1', "io.acryl:datahub-spark-lineage:0.8.23"]).getOrCreate()

In [ ]:
jar_packages = ["org.apache.hadoop:hadoop-aws:3.2.3", "io.delta:delta-core_2.12:1.2.1", "io.acryl:datahub-spark-lineage:0.8.23"]
jar_packages = ["org.apache.hadoop:hadoop-aws:3.3.1", "io.delta:delta-spark_2.12:3.0.0", "io.acryl:datahub-spark-lineage:0.8.23"]
spark = SparkSession.builder \
    .appName("exampleOpenSanctionsDatahub") \
    .master("local[*]") \
    .config("spark.jars.packages", ",".join(jar_packages)) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.extraListeners","datahub.spark.DatahubSparkListener") \
    .config("spark.datahub.rest.server", "http://localhost:8080") \
    .enableHiveSupport() \
    .getOrCreate()

In [ ]:
file_path = "s3a://warehouse/files/opensanctions.file/entities.ftm.json"
delta_table_path = "s3a://warehouse/bronze/opensanctions_entities.delta"

In [ ]:
# Read file into a DataFrame
df = spark.read.json(file_path)

In [ ]:
df.show()

In [ ]:
# Write DataFrame to Delta table
df.write.format("delta").mode("overwrite").save(delta_table_path)

# Stop the Spark session
spark.stop()

# Processing

Processing these step before writing data to layer silver
1. Transform to standard schema of layer silver
2. Unique each record
3. Add fields
4. Map entities
5. Upsert

Read delta table and discovery data

In [ ]:
# Create a Spark session
spark = SparkSession.builder \
    .appName("JsonToDelta") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.0.0,org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", obj_storage_endpoint) \
    .config("spark.hadoop.fs.s3a.access.key", obj_storage_access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", obj_storage_secret_key) \
    .config('spark.hadoop.fs.s3a.path.style.access', "true") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

In [ ]:
delta_table_path = "s3a://warehouse/bronze/opensanctions_entities.delta"

In [ ]:
# Read Delta table
df = spark.read.format("delta").load(delta_table_path)

In [ ]:
df.show()